[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day5_solution.ipynb)

# Day 5 · 정답 — 사진을 읽는 신경망

합성곱으로 만들고 남이 배운 것을 가져온다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`lecture` 와 `practice` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 필터를 직접 통과시켜 본다

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(42)

train = datasets.CIFAR10('data', train=True,  download=True, transform=transforms.ToTensor())
test  = datasets.CIFAR10('data', train=False, download=True, transform=transforms.ToTensor())

# 5,000장만 떼어 쓴다. Subset 은 데이터에서 일부만 골라 주는 것이다.
# 전체 5만 장으로 돌리면 한 번에 몇 분씩 걸려 여러 번 비교하기 어렵다.
small      = Subset(train, range(5000))
small_test = Subset(test,  range(1000))
loader      = DataLoader(small,      batch_size=128, shuffle=True)
test_loader = DataLoader(small_test, batch_size=500)

names = train.classes
print(names)

In [ ]:
def fit(model, ld=None, epochs=6, lr=0.001):
    """학습 루프 다섯 줄을 함수로 묶어 둔 것 — 1주차에 만든 것과 같다"""
    ld = ld if ld is not None else loader
    torch.manual_seed(42)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in ld:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model

def score(model, ld=None):
    """학습에 안 쓴 자료로 재는 정확도"""
    ld = ld if ld is not None else test_loader
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in ld:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total

def count(model):
    """배울 계수가 몇 개인지"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
img, label = test[13]
print('모양   ', img.shape)
print('이름   ', names[label])
print('값 범위', float(img.min()), '~', float(img.max()))

plt.imshow(img.permute(1, 2, 0))   # imshow 는 [행, 열, 색] 순서를 원한다
plt.axis('off'); plt.show()

# 색 세 면의 평균으로 흑백 한 장을 만들어 둔다. 필터 실습에서 이것을 쓴다.
gray = img.mean(0)
print('흑백 모양', gray.shape)

In [ ]:
def apply_filter(k, image=None):
    """3x3 필터 하나를 흑백 사진에 통과시켜 결과를 돌려준다"""
    image = image if image is not None else gray
    c = nn.Conv2d(1, 1, 3, padding=1, bias=False)
    c.weight.data[0, 0] = torch.tensor(k, dtype=torch.float)
    with torch.no_grad():
        return c(image.view(1, 1, 32, 32))[0, 0]

vert = apply_filter([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
print('세로 윤곽선 범위 %+.0f ~ %+.0f' % (vert.min() * 255, vert.max() * 255))

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 1.** `img` 에서 **빨강 면 하나만** 꺼내 `red` 에 담는다. 모양이 `[3, 32, 32]` 이고 0번 축이 색이므로 그 축의 0번을 고른다.

In [ ]:
red = img[0]

print(red.shape)
assert tuple(red.shape) == (32, 32), f'32x32 여야 한다: {tuple(red.shape)}'
plt.imshow(red, cmap='Reds'); plt.axis('off'); plt.show()

> **실습문제 2.** 컬러 사진(채널 3)을 받아 **특징 지도 32장**을 내놓는 합성곱 층을 만든다. 창은 3×3, `padding=1` 로 가로세로를 유지한다.

In [ ]:
conv = nn.Conv2d(3, 32, 3, padding=1)

out = conv(img.unsqueeze(0))   # unsqueeze(0) 은 사진 한 장을 배치 하나로 감싸는 것
print(out.shape)
assert tuple(out.shape) == (1, 32, 32, 32), f'[1,32,32,32] 여야 한다: {tuple(out.shape)}'

> **실습문제 3.** `padding` 을 **0** 으로 바꾸면 가로세로가 어떻게 되는지 확인한다. 창이 사진 안에만 들어가야 하므로 양쪽 한 줄씩 못 쓴다.

In [ ]:
c0 = nn.Conv2d(3, 8, 3, padding=0)
print(c0(img.unsqueeze(0)).shape)

s = tuple(c0(img.unsqueeze(0)).shape)
assert s == (1, 8, 30, 30), f'[1,8,30,30] 이어야 한다: {s}'
print('32 였던 것이', s[2], '가 됐다')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** **가로 윤곽선**을 찾는 필터를 적어 `horz` 에 담는다. 세로 필터가 좌우로 `-1 0 +1` 이었으니, 가로는 **위아래로** 같은 모양을 쓴다.

In [ ]:
horz = apply_filter([[-1, -1, -1],
                     [ 0,  0,  0],
                     [ 1,  1,  1]])

print(horz.shape)
assert tuple(horz.shape) == (32, 32)
assert horz.abs().max() > 0.3, '값이 거의 0이면 필터가 잘못 적힌 것이다'

> **실습문제 4.** 세로 · 가로 · 흐리게 세 결과를 원본과 함께 **한 줄에 네 장**으로 그린다. 흐리게 필터는 아홉 칸 모두 `1/9` 이다. 그림 제목에 무엇인지 적는다.

In [ ]:
blur = apply_filter([[1/9] * 3] * 3)

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, (t, im) in zip(axes, [('원본', gray), ('세로', vert.abs()),
                              ('가로', horz.abs()), ('흐리게', blur)]):
    ax.imshow(im, cmap='gray')
    ax.set_title(t); ax.axis('off')
plt.show()
assert 'blur' in dir(), 'blur 를 만들어야 한다'
print('세로가 남긴 양 %.1f · 가로가 남긴 양 %.1f' % (vert.abs().sum(), horz.abs().sum()))

## 2. shape 를 따라간다

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 5.** `Conv2d` → `ReLU` → `MaxPool2d` 한 묶음을 통과시킨 뒤 모양을 찍는다. 채널은 3에서 16으로, 가로세로는 32에서 절반이 된다.

In [ ]:
block = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2))
print(block(img.unsqueeze(0)).shape)

s = tuple(block(img.unsqueeze(0)).shape)
assert s == (1, 16, 16, 16), f'[1,16,16,16] 이어야 한다: {s}'

> **실습문제 6.** `MaxPool2d` 를 빼고 `Flatten` 뒤에 `Linear(64*8*8, 10)` 을 붙이면 에러가 난다. 일부러 내 보고 **메시지의 네 숫자**를 읽는다.

In [ ]:
bad = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
try:
    bad(img.unsqueeze(0))
except RuntimeError as e:
    msg = str(e)
    print(msg)

assert 'msg' in dir() and 'shapes cannot be multiplied' in msg, '에러가 나야 정상이다'
print('들어온 칸 수 65536 · 적어 둔 칸 수 4096 — 풀링을 빼서 절반으로 줄지 않았다')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 7.** 묶음을 **세 번** 쌓으면 가로세로가 몇이 되는지 코드로 확인한다. 채널은 3 → 16 → 32 → 64 로 두껍게 만든다. 마지막에 `Flatten()` 을 붙여 **몇 칸이 되는지** 찍는다.

In [ ]:
deep = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten())
print(deep(img.unsqueeze(0)).shape)
n = deep(img.unsqueeze(0)).shape[1]
assert n == 64 * 4 * 4, f'64*4*4 = 1024 여야 한다: {n}'
print('32 → 16 → 8 → 4 이므로 64 x 4 x 4 =', n)

## 3. CNN 을 만들어 돌린다

In [ ]:
torch.manual_seed(42)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(3072, 128), nn.ReLU(), nn.Linear(128, 10))
fit(mlp)
print('펴서 Linear  정확도 %.4f  계수 %d' % (score(mlp), count(mlp)))

# 비교 기준이 될 CNN 도 미리 한 벌 만들어 둔다. 아래 문제들이 이것과 견준다.
torch.manual_seed(42)
cnn = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
fit(cnn)
print('CNN          정확도 %.4f  계수 %d' % (score(cnn), count(cnn)))

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 8.** 준비 셀의 CNN 을 **직접 다시 써서** `my_cnn` 을 만든다. 채널은 3 → 32 → 64, 가로세로는 32 → 16 → 8 이므로 마지막 `Linear` 의 입력은 `64*8*8` 이다. 학습까지 시켜 준비 셀과 같은 값이 나오는지 본다.

In [ ]:
torch.manual_seed(42)
my_cnn = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
fit(my_cnn)
print('정확도 %.4f  계수 %d' % (score(my_cnn), count(my_cnn)))

assert count(my_cnn) == 60362, f'계수가 60,362 여야 한다: {count(my_cnn)}'
assert score(my_cnn) == score(cnn), '준비 셀과 같은 값이 나와야 한다'
print('계수는 %d 분의 1 인데 정확도는 올랐다' % (count(mlp) // count(my_cnn)))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 9.** 묶음 **하나**짜리 CNN 도 만들어 셋을 나란히 비교한다. `계수 · 정확도` 를 함께 찍는다. 묶음이 하나면 가로세로가 16 이므로 `Linear` 의 입력은 `32*16*16` 이다.

In [ ]:
torch.manual_seed(42)
one = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 16 * 16, 10))
fit(one)

for t, m in [('펴서 Linear', mlp), ('묶음 하나', one), ('묶음 둘', cnn)]:
    print('%-12s 계수 %7d   정확도 %.4f' % (t, count(m), score(m)))
assert 'one' in dir(), 'one 을 만들어야 한다'
print('묶음 하나가 계수는 더 많다 — 펴는 칸이 8192 라서다')
print('정확도는 둘이 거의 같다. 5,000장에서는 깊이의 값이 아직 안 나온다')

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 10.** 필터 장수를 **16장**과 **64장**으로 바꿔 묶음 둘짜리 CNN 을 두 개 더 만든다. `계수 · 정확도 · 걸린 시간` 세 값을 표로 찍는다. 시간은 `time.time()` 으로 잰다.

In [ ]:
import time
rows = []
for k in (16, 32, 64):
    torch.manual_seed(42)
    m = nn.Sequential(
        nn.Conv2d(3, k, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(k, k * 2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(k * 2 * 8 * 8, 10))
    t0 = time.time()
    fit(m)
    rows.append((k, count(m), score(m), time.time() - t0))

for k, n, a, s in rows:
    print('필터 %2d장  계수 %7d  정확도 %.4f  %4.0f초' % (k, n, a, s))
assert len(rows) == 3, '세 줄이 나와야 한다'
assert rows[0][1] < rows[2][1], '필터가 많으면 계수도 많다'

> **실습문제 11.** 가장 많이 **틀린 두 종류**를 찾는다. 종류마다 몇 장 중 몇 장을 맞혔는지 세어 정확도가 낮은 순으로 찍는다.

In [ ]:
hit = [0] * 10
tot = [0] * 10
cnn.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        pred = cnn(xb).argmax(1)
        for i in range(len(yb)):
            tot[yb[i]] += 1
            hit[yb[i]] += int(pred[i] == yb[i])

rank = sorted(range(10), key=lambda c: hit[c] / tot[c])
for c in rank:
    print('%-10s %3d / %3d   %.3f' % (names[c], hit[c], tot[c], hit[c] / tot[c]))
assert sum(tot) == 1000, f'1,000장을 다 세야 한다: {sum(tot)}'
print('가장 어려운 둘:', names[rank[0]], '·', names[rank[1]])

## 4. 이미 배운 모델을 가져온다

In [ ]:
from torchvision import models

net = models.resnet18(weights='DEFAULT')
print('전체 계수', sum(p.numel() for p in net.parameters()))
print('마지막 층 ', net.fc)

In [ ]:
T224 = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

big_train = datasets.CIFAR10('data', train=True,  transform=T224)
big_test  = datasets.CIFAR10('data', train=False, transform=T224)

backbone = models.resnet18(weights='DEFAULT')
backbone.fc = nn.Identity()      # 마지막 층을 없애는 대신 그대로 내놓게 한다
backbone.eval()

def features(ds, n):
    """사진 n 장을 얼린 앞쪽에 한 번 통과시켜 512칸씩 뽑아 둔다"""
    X, Y = [], []
    with torch.no_grad():
        for xb, yb in DataLoader(Subset(ds, range(n)), batch_size=64):
            X.append(backbone(xb)); Y.append(yb)
    return torch.cat(X), torch.cat(Y)

# 6,000장을 224x224 로 한 번 통과시킨다. GPU 면 30초, CPU 면 몇 분 걸린다.
# 런타임 유형을 GPU 로 바꿔 두면 훨씬 빠르다.
Xtr, Ytr = features(big_train, 5000)
Xte, Yte = features(big_test,  1000)
print(Xtr.shape, Xte.shape)

In [ ]:
def train_head(X, Y, epochs=30, Xv=None, Yv=None):
    """512칸으로 Linear(512, 10) 하나만 학습시켜 (모델, 정확도) 를 돌려준다"""
    Xv = Xv if Xv is not None else Xte
    Yv = Yv if Yv is not None else Yte
    torch.manual_seed(42)
    h = nn.Linear(512, 10)
    o = torch.optim.Adam(h.parameters(), lr=0.001)
    f = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 128):
            b = perm[i:i + 128]
            o.zero_grad()
            f(h(X[b]), Y[b]).backward()
            o.step()
    with torch.no_grad():
        return h, (h(Xv).argmax(1) == Yv).float().mean().item()

head, acc = train_head(Xtr, Ytr)
print('전이학습 정확도 %.4f  학습한 계수 %d' % (acc, count(head)))

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 12.** 마지막 층을 **10 종류짜리**로 갈아 끼운다. 입력 칸 수 `512` 는 앞쪽 층이 내놓는 값이라 바꿀 수 없다.

In [ ]:
net.fc = nn.Linear(512, 10)
print(net.fc)
print('새 층의 계수', sum(p.numel() for p in net.fc.parameters()))

n = sum(p.numel() for p in net.fc.parameters())
assert n == 5130, f'512*10+10 = 5,130 이어야 한다: {n}'

> **실습문제 13.** 준비 셀의 `train_head` 안쪽 루프를 **직접 써서** `my_head` 를 학습시킨다. 1주차 학습 루프 다섯 줄과 같고, 사진 대신 `Xtr` 의 512칸을 먹인다는 점만 다르다.

In [ ]:
torch.manual_seed(42)
my_head = nn.Linear(512, 10)
opt = torch.optim.Adam(my_head.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for e in range(30):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):
        b = perm[i:i + 128]
        opt.zero_grad()                          # 기울기를 비운다
        loss_fn(my_head(Xtr[b]), Ytr[b]).backward()   # 기울기를 구한다
        opt.step()                          # 한 걸음 옮긴다

with torch.no_grad():
    my_acc = (my_head(Xte).argmax(1) == Yte).float().mean().item()
print('%.4f  학습한 계수 %d' % (my_acc, count(my_head)))

assert count(my_head) == 5130, f'512*10+10 = 5,130 이어야 한다: {count(my_head)}'
assert abs(my_acc - acc) < 1e-6, f'준비 셀과 같아야 한다: {my_acc} 대 {acc}'
print('계수 5,130개로 %.4f — CNN 60,362개보다 적게 배우고 더 맞혔다' % my_acc)

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 14.** 훈련 장수를 **5,000 · 2,000 · 500 · 200** 으로 줄여 가며 전이학습 정확도를 잰다. 이미 뽑아 둔 `Xtr` 의 앞쪽만 잘라 쓰면 특징을 다시 뽑을 필요가 없다.

In [ ]:
for n in (5000, 2000, 500, 200):
    _, a = train_head(Xtr[:n], Ytr[:n])
    print('훈련 %4d장  전이학습 %.4f' % (n, a))
_, a200 = train_head(Xtr[:200], Ytr[:200])
print('5,000장 %.4f → 200장 %.4f · 25분의 1로 줄여도 얼마나 버티는지 본다' % (acc, a200))

> **빈칸 문제 2.** 정규화를 **빼면** 얼마나 떨어지는지 본다. `Resize` 와 `ToTensor` 만 쓴 형식으로 특징을 다시 뽑아 같은 방식으로 학습한다. 에러는 나지 않으니 숫자로만 알 수 있다.

In [ ]:
T_no = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor()])          # Normalize 를 뺐다

raw_train = datasets.CIFAR10('data', train=True,  transform=T_no)
raw_test  = datasets.CIFAR10('data', train=False, transform=T_no)
Xr, Yr = features(raw_train, 5000)
Xs, Ys = features(raw_test,  1000)

_, bad_acc = train_head(Xr, Yr, Xv=Xs, Yv=Ys)
print('정규화 있음 %.4f · 없음 %.4f' % (acc, bad_acc))
assert bad_acc < acc, '정규화를 빼면 떨어져야 한다'

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 15.** 같은 **200장**으로 CNN 을 처음부터 학습시켜 전이학습과 나란히 비교한다. `Subset(train, range(200))` 으로 작은 loader 를 만들어 쓴다.

In [ ]:
tiny = DataLoader(Subset(train, range(200)), batch_size=32, shuffle=True)
torch.manual_seed(42)
scratch = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 10))
fit(scratch, tiny, epochs=20)
print('200장 · CNN 처음부터   %.4f  계수 %d' % (score(scratch), count(scratch)))
print('200장 · 전이학습       %.4f  계수 %d' % (train_head(Xtr[:200], Ytr[:200])[1], 5130))
assert 'scratch' in dir(), 'scratch 를 만들어야 한다'
print('사진이 적을 때 어느 쪽이 쓸 만한지가 이 실습의 결론이다')

## 5. 세 방식을 한 표로

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 16.** 오늘 만든 세 방식을 **한 표**로 정리해 찍는다. `방식 · 학습한 계수 · 정확도` 세 칸이다. 전이학습은 테스트 500장, 나머지는 1,000장이라 조건이 다른 것을 표 아래에 적는다.

In [ ]:
print('%-22s %10s %10s' % ('방식', '학습 계수', '정확도'))
print('-' * 44)
print('%-22s %10d %10.4f' % ('펴서 Linear', count(mlp), score(mlp)))
print('%-22s %10d %10.4f' % ('CNN 처음부터', count(cnn), score(cnn)))
print('%-22s %10d %10.4f' % ('resnet18 특징 + Linear', 5130, acc))
print()
print('훈련 5,000장 · 테스트 1,000장 — 세 방식 모두 같은 조건이다')
print('계수는 줄고 정확도는 오른다 — 이것이 오늘의 결론이다')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 17.** 전이학습이 **틀린 사진 여섯 장**을 골라 `실제 → 예측` 을 제목으로 붙여 그린다. `big_test` 의 사진은 정규화돼 있어 그대로 그리면 색이 이상하니 `test` 에서 같은 번호를 가져온다.

In [ ]:
with torch.no_grad():
    pred = head(Xte).argmax(1)

wrong = [i for i in range(len(Yte)) if pred[i] != Yte[i]][:6]

fig, axes = plt.subplots(1, 6, figsize=(13, 3))
for ax, i in zip(axes, wrong):
    ax.imshow(test[i][0].permute(1, 2, 0))
    ax.set_title('%s → %s' % (names[Yte[i]], names[pred[i]]), fontsize=9)
    ax.axis('off')
plt.show()
assert len(wrong) == 6, f'여섯 장을 골라야 한다: {len(wrong)}'
for i in wrong:
    print('%-10s 인데 %s 라고' % (names[Yte[i]], names[pred[i]]))

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 18.** 오늘 새로 배운 다섯 줄을 각각 **한 문장으로** 설명하는 주석을 달아 셀에 적는다. `nn.Conv2d` · `nn.MaxPool2d` · `models.resnet18` · `requires_grad` · `net.fc` 대입.

In [ ]:
# nn.Conv2d(3, 32, 3, padding=1)
#   3x3 창을 미끄러뜨려 무늬를 찾는다. 채널이 3에서 32로 두꺼워진다.
# nn.MaxPool2d(2)
#   겹치지 않는 2x2 마다 최댓값 하나만 남긴다. 가로세로가 절반이 된다.
# models.resnet18(weights='DEFAULT')
#   사진 128만 장으로 학습이 끝난 계수까지 함께 받아 온다.
# p.requires_grad = False
#   그 계수의 기울기를 구하지 않는다. 학습에서 움직이지 않는다.
# net.fc = nn.Linear(512, 10)
#   마지막 층만 내 문제 크기로 갈아 끼운다. 앞쪽은 그대로 쓴다.
print('정리 끝')
print('이 다섯 줄이 오늘 늘어난 전부다')